In [1]:
import sys
sys.path.append("..")

import json
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data import load_model_data, SEED, RESULTS_DIR
from src.models import Model, MODEL_NAMES, PARAM_GRIDS
from src.evaluation import fit_evaluate

X, y, _, _ = load_model_data()
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
METRICS = ["macro_f1", "balanced_accuracy", "accuracy", "detection_rate", "false_alarm_rate", "fit_s"]
print(X.shape, X_tr.shape, X_val.shape)

(107740, 42) (86192, 42) (21548, 42)


In [2]:
rows = []
for name in MODEL_NAMES:
    for params in PARAM_GRIDS[name]:
        scores, _, _ = fit_evaluate(Model(name, params), X_tr, y_tr, X_val, y_val)
        rows.append({"model": name, "params": json.dumps(params), **{m: scores[m] for m in METRICS}})
        print(name, params, round(scores["macro_f1"], 4), round(scores["fit_s"], 1))
tuning = pd.DataFrame(rows)
tuning.to_csv(RESULTS_DIR / "02_tuning.csv", index=False)

Dummy {} 0.065 0.4


LogReg {'C': 0.1} 0.4589 45.4


LogReg {'C': 1.0} 0.4628 79.1


LogReg {'C': 10.0} 0.4629 136.6


kNN {'n_neighbors': 1, 'weights': 'uniform'} 0.48 0.4


...

RandomForest {'max_features': 0.3, 'min_samples_leaf': 3} 0.6117 39.3


XGBoost {'max_depth': 6, 'learning_rate': 0.1} 0.6368 54.5


XGBoost {'max_depth': 6, 'learning_rate': 0.3} 0.6326 55.0


XGBoost {'max_depth': 10, 'learning_rate': 0.1} 0.6301 81.0


XGBoost {'max_depth': 10, 'learning_rate': 0.3} 0.6183 75.3


In [3]:
selected_rows = tuning.loc[tuning.groupby("model", sort=False)["macro_f1"].idxmax()].set_index("model")
selected = {name: json.loads(selected_rows.loc[name, "params"]) for name in MODEL_NAMES}
(RESULTS_DIR / "02_selected_params.json").write_text(json.dumps(selected, indent=2))
selected_rows.round(4)

,params,macro_f1,balanced_accuracy,accuracy,detection_rate,false_alarm_rate,fit_s
model,,,,,,,
Dummy,{},0.0650,0.1000,0.4816,0.0000,0.0000,0.3634
LogReg,"{""C"": 10.0}",0.4629,0.6196,0.6926,0.9985,0.1949,136.5717
kNN,"{""n_neighbors"": 5, ""weights"": ""distance""}",0.4824,0.4755,0.7804,0.9225,0.0815,0.4945
NaiveBayes,"{""var_smoothing"": 0.001}",0.2301,0.4069,0.4725,0.9799,0.2667,0.8141
RandomForest,"{""max_features"": 0.3, ""min_samples_leaf"": 1}",0.6203,0.6392,0.8040,0.9918,0.1526,42.5514
XGBoost,"{""max_depth"": 6, ""learning_rate"": 0.1}",0.6368,0.6938,0.7997,0.9927,0.1592,54.5130


In [4]:
variants = [
    ("LogReg", {"log": False}), ("kNN", {"log": False}), ("NaiveBayes", {"log": False}),
    ("LogReg", {"balanced": False}), ("NaiveBayes", {"balanced": False}),
    ("RandomForest", {"balanced": False}), ("XGBoost", {"balanced": False}),
    ("XGBoost", {"smote": True}),
]
rows = []
for name, option in variants:
    scores, _, _ = fit_evaluate(Model(name, selected[name], **option), X_tr, y_tr, X_val, y_val)
    (key, value), = option.items()
    rows.append({"model": name, "variant": f"{key}={value}", **{m: scores[m] for m in METRICS}})
design = pd.DataFrame(rows)
for m in ["macro_f1", "balanced_accuracy", "detection_rate", "false_alarm_rate"]:
    design[f"delta_{m}"] = design[m] - design["model"].map(selected_rows[m])
design.to_csv(RESULTS_DIR / "02_design_checks.csv", index=False)
design.round(4)

,model,variant,macro_f1,balanced_accuracy,accuracy,detection_rate,false_alarm_rate,fit_s,delta_macro_f1,delta_balanced_accuracy,delta_detection_rate,delta_false_alarm_rate
0,LogReg,log=False,0.4470,0.5993,0.6911,0.9981,0.1938,120.0697,-0.0159,-0.0203,-0.0004,-0.0012
1,kNN,log=False,0.4249,0.4167,0.7362,0.9079,0.1047,0.4425,-0.0574,-0.0588,-0.0146,0.0232
2,NaiveBayes,log=False,0.2599,0.4163,0.4832,0.9731,0.2545,0.8977,0.0298,0.0093,-0.0069,-0.0122
3,LogReg,balanced=False,0.4457,0.4375,0.7598,0.9344,0.1162,74.2336,-0.0172,-0.1821,-0.0641,-0.0787
4,NaiveBayes,balanced=False,0.2320,0.4097,0.4832,0.9764,0.2465,0.7338,0.0019,0.0027,-0.0036,-0.0202
5,RandomForest,balanced=False,0.5821,0.5709,0.8214,0.9453,0.0726,62.8144,-0.0382,-0.0683,-0.0465,-0.0801
6,XGBoost,balanced=False,0.6116,0.6044,0.8261,0.9456,0.0732,60.0167,-0.0252,-0.0895,-0.0472,-0.0860
7,XGBoost,smote=True,0.5953,0.6671,0.8090,0.9525,0.0972,666.4674,-0.0415,-0.0267,-0.0403,-0.0620
